# V4 Prior Engine: Continuous xG Dixon-Coles Optimizer

This notebook implements the foundational Module 1 of the V4 Bayesian Tri-State architecture.
Instead of training on stochastic discrete goals, it optimizes Attack ($\alpha$) and Defense ($\beta$) parameters by minimizing the Negative Log-Likelihood of **Continuous Expected Goals (xG)**.

Crucially, it implements:
- Exponential Time-Decay (recent matches weigh heavier).
- Optimization constraints ($\frac{1}{N} \sum \alpha = 1.0$) to prevent infinite parameter scaling.
- The $\rho$ interdependence factor trained on discrete goals to capture the human behavioral element of 0-0 and 1-1 draws.

In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from datetime import datetime, timedelta

np.random.seed(42)


## 1. Generate the Historical Match Ledger
We generate 3 seasons worth of data for a 20-team league to stress-test the optimizer.

In [3]:
teams = [f"Team_{i}" for i in range(1, 21)]
n_teams = len(teams)
team_to_idx = {team: i for i, team in enumerate(teams)}

# Create underlying "true" strengths to see if the optimizer can recover them
true_alpha = np.random.uniform(0.7, 1.5, n_teams)
true_beta = np.random.uniform(0.7, 1.3, n_teams)
true_gamma = 1.15 # Home advantage

matches = []
current_date = datetime(2026, 9, 1)

# Simulate 3 seasons (approx 1095 days)
for day_offset in range(1095):
    if day_offset % 3 != 0: continue # Games every ~3 days
    
    match_date = current_date - timedelta(days=1095 - day_offset)
    
    # Pick two random teams
    home, away = np.random.choice(teams, 2, replace=False)
    home_i, away_i = team_to_idx[home], team_to_idx[away]
    
    # Calculate true expected xG
    lambda_val = true_alpha[home_i] * true_beta[away_i] * true_gamma
    mu_val = true_alpha[away_i] * true_beta[home_i]
    
    # Add noise to simulate real-world xG recording
    home_xg = max(0.1, np.random.normal(lambda_val, 0.4))
    away_xg = max(0.1, np.random.normal(mu_val, 0.4))
    
    # Generate discrete goals via Poisson (to train the rho parameter later)
    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)
    
    matches.append({
        "date": match_date,
        "days_ago": 1095 - day_offset,
        "home_team": home_i,
        "away_team": away_i,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_goals": home_goals,
        "away_goals": away_goals
    })

df_matches = pd.DataFrame(matches)
print(f"Generated {len(df_matches)} historical matches across 3 seasons.")
display(df_matches.head())


Generated 365 historical matches across 3 seasons.


,date,days_ago,home_team,away_team,home_xg,away_xg,home_goals,away_goals
0,2023-09-02,1095,15,16,1.154966,1.186505,0,0
1,2023-09-05,1092,1,5,2.221578,0.283701,6,0
2,2023-09-08,1089,6,3,0.729391,0.580266,0,1
3,2023-09-11,1086,7,9,1.268809,1.403080,0,4
4,2023-09-14,1083,9,13,1.951708,0.603375,2,0


## 2. The Negative Log-Likelihood Optimizer
We drop the $x!$ and $y!$ denominators as they are constants relative to our parameters $\alpha, \beta, \gamma$.

In [4]:
def rho_correction(x_goals, y_goals, lambda_val, mu_val, rho):
    """
    Dixon-Coles interdependence correction. 
    Uses integer goals, NOT xG, because the 0-0 inflation is a human psychological phenomenon.
    """
    if x_goals == 0 and y_goals == 0:
        return 1.0 - (lambda_val * mu_val * rho)
    elif x_goals == 0 and y_goals == 1:
        return 1.0 + (lambda_val * rho)
    elif x_goals == 1 and y_goals == 0:
        return 1.0 + (mu_val * rho)
    elif x_goals == 1 and y_goals == 1:
        return 1.0 - rho
    else:
        return 1.0

def continuous_dc_log_likelihood(params, df, xi_decay):
    """
    params array: 
    [0:20] -> alphas (Attack)
    [20:40] -> betas (Defense)
    [40] -> gamma (Home Advantage)
    [41] -> rho (Low-score interdependence)
    """
    alphas = params[:20]
    betas = params[20:40]
    gamma = params[40]
    rho = params[41]
    
    h_idx = df['home_team'].values
    a_idx = df['away_team'].values
    
    # Continuous historical xG
    home_xg = df['home_xg'].values
    away_xg = df['away_xg'].values
    
    # Discrete goals for rho correction
    home_goals = df['home_goals'].values
    away_goals = df['away_goals'].values
    
    # Time decay weights (w_m = e^{-xi * t_m})
    weights = np.exp(-xi_decay * df['days_ago'].values)
    
    # Expected Goals based on parameters
    lambdas = alphas[h_idx] * betas[a_idx] * gamma
    mus = alphas[a_idx] * betas[h_idx]
    
    # Vectorized Rho Correction
    tau = np.ones(len(df))
    mask_00 = (home_goals == 0) & (away_goals == 0)
    mask_01 = (home_goals == 0) & (away_goals == 1)
    mask_10 = (home_goals == 1) & (away_goals == 0)
    mask_11 = (home_goals == 1) & (away_goals == 1)
    
    tau[mask_00] = 1.0 - (lambdas[mask_00] * mus[mask_00] * rho)
    tau[mask_01] = 1.0 + (lambdas[mask_01] * rho)
    tau[mask_10] = 1.0 + (mus[mask_10] * rho)
    tau[mask_11] = 1.0 - rho
    
    # Ensure tau doesn't push log into negatives due to extreme parameters
    tau = np.clip(tau, 1e-6, None)
    
    # The Continuous Log-Likelihood objective function
    # LL = (x_hat * ln(lambda) - lambda) + (y_hat * ln(mu) - mu) + ln(tau)
    ll = (home_xg * np.log(lambdas) - lambdas) + (away_xg * np.log(mus) - mus) + np.log(tau)
    
    # Apply time decay weight and sum
    weighted_ll = np.sum(ll * weights)
    
    # We want to MAXIMIZE log-likelihood, so we MINIMIZE negative log-likelihood
    return -weighted_ll


## 3. Apply Constraints & Execute Optimization

In [5]:
# Initial Guess: 1.0 for all parameters, 1.15 for gamma, 0.0 for rho
initial_guess = np.concatenate([np.ones(40), [1.15, 0.0]])

# Constraint: The average of all attack parameters (alphas) must exactly equal 1.0
def alpha_constraint(params):
    return np.mean(params[:20]) - 1.0

constraints = [{'type': 'eq', 'fun': alpha_constraint}]

# Bounds: Prevent parameters from collapsing to zero or exploding to infinity
# Alphas & Betas [0.1, 3.0], Gamma [0.5, 2.0], Rho [-0.2, 0.2]
bounds = [(0.1, 3.0)] * 40 + [(0.5, 2.0), (-0.2, 0.2)]

print("Starting Optimization Solver...")
xi_decay = 0.0015 # Standard half-life decay

opt_res = minimize(
    continuous_dc_log_likelihood, 
    initial_guess, 
    args=(df_matches, xi_decay), 
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 500, 'disp': True}
)

if opt_res.success:
    print("\n✅ Optimization Converged Successfully!")
    estimated_alphas = opt_res.x[:20]
    estimated_betas = opt_res.x[20:40]
    estimated_gamma = opt_res.x[40]
    estimated_rho = opt_res.x[41]
    
    print(f"\nEstimated Home Advantage (Gamma): {estimated_gamma:.3f}")
    print(f"Estimated Rho factor: {estimated_rho:.3f}")
    print(f"Alpha Array Mean Constraint Check: {np.mean(estimated_alphas):.5f}")
else:
    print("❌ Optimization Failed:", opt_res.message)


Starting Optimization Solver...
Optimization terminated successfully    (Exit mode 0)
            Current function value: 337.29971812436634
            Iterations: 25
            Function evaluations: 1103
            Gradient evaluations: 25

✅ Optimization Converged Successfully!

Estimated Home Advantage (Gamma): 1.182
Estimated Rho factor: -0.082
Alpha Array Mean Constraint Check: 1.00000
